In [170]:
import re
import math
from urllib.parse import urlparse
from collections import Counter
import pandas as pd

In [171]:
def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0
    freq = Counter(s)
    length = len(s)
    entropy = 0.0
    for count in freq.values():
        p = count / length
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy

def _count_digits(s: str) -> int:
    return sum(1 for c in s if c.isdigit())

In [172]:
def extract_features(url: str) -> list[float]:
    url = url.strip()
    url_len = len(url)

    if url_len == 0:
        raise ValueError("Empty URL")

    letter_cnt = sum(1 for c in url if c.isalpha())
    digit_cnt = sum(1 for c in url if c.isdigit())
    special_cnt = url_len - letter_cnt - digit_cnt

    eq_cnt = url.count("=")
    qm_cnt = url.count("?")
    amp_cnt = url.count("&")
    dot_cnt = url.count(".")
    dash_cnt = url.count("-")
    under_cnt = url.count("_")
    slash_cnt = url.count("/")

    letter_ratio = letter_cnt / url_len
    digit_ratio = digit_cnt / url_len
    spec_ratio = special_cnt / url_len

    is_https = 1 if url.lower().startswith("https://") else 0

    try:
        parsed = urlparse(url)
        hostname = parsed.hostname or ""
        pathname = parsed.path or ""
        query = parsed.query or ""
    except:
        hostname, pathname, query = "", url, ""

    host_lower = hostname.lower()

    has_www = 1 if host_lower.startswith("www.") else 0
    host_clean = host_lower[4:] if has_www else host_lower

    parts = host_clean.split(".") if host_clean else []

    is_ip = bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host_clean))

    if is_ip or len(parts) < 2:
        tld_len = 0
        dom_len = len(host_clean)
        subdom_cnt = 0
        dom_digits = 0
        dom_digit_ratio = 0.0
    else:
        tld_len = len(parts[-1])

        dom_parts = parts[-2:]
        dom_str = ".".join(dom_parts)
        dom_len = len(dom_str)
        subdom_cnt = max(0, len(parts) - 2)

        dom_digits = _count_digits(dom_str)
        dom_digit_ratio = dom_digits / dom_len if dom_len > 0 else 0.0

    path_len = len(pathname)
    query_len = len(query)

    path_digits = _count_digits(pathname)
    query_digits = _count_digits(query)

    path_digit_ratio = path_digits / path_len if path_len > 0 else 0.0
    query_digit_ratio = query_digits / query_len if query_len > 0 else 0.0

    entropy = shannon_entropy(url)

    return [
        float(url_len),
        float(dom_len),
        float(is_ip),
        float(tld_len),
        float(subdom_cnt),
        float(letter_cnt),
        float(digit_cnt),
        float(special_cnt),
        float(eq_cnt),
        float(qm_cnt),
        float(amp_cnt),
        float(dot_cnt),
        float(dash_cnt),
        float(under_cnt),
        letter_ratio,
        digit_ratio,
        spec_ratio,
        float(is_https),
        float(slash_cnt),
        entropy,
        float(path_len),
        float(query_len),
        float(has_www),
        dom_digit_ratio,
        path_digit_ratio,
        query_digit_ratio,
    ]

In [173]:
full_feature_names = [
    "url_len",
    "dom_len",
    "is_ip",
    "tld_len",
    "subdom_cnt",
    "letter_cnt",
    "digit_cnt",
    "special_cnt",
    "eq_cnt",
    "qm_cnt",
    "amp_cnt",
    "dot_cnt",
    "dash_cnt",
    "under_cnt",
    "letter_ratio",
    "digit_ratio",
    "spec_ratio",
    "is_https",
    "slash_cnt",
    "entropy",
    "path_len",
    "query_len",
    "has_www",
    "dom_digit_ratio",
    "path_digit_ratio",
    "query_digit_ratio",
]

In [174]:
def extract_domain_features(url: str) -> list[float]:
    try:
        parsed = urlparse(url)
        hostname = parsed.hostname or ""
    except:
        hostname = ""

    if not hostname:
        raise ValueError(f"Invalid URL: {url}")

    host_lower = hostname.lower()

    has_www = 1 if host_lower.startswith("www.") else 0
    host_clean = host_lower[4:] if has_www else host_lower

    entropy = shannon_entropy(host_clean)

    is_ip = 1 if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host_clean) else 0

    parts = host_clean.split(".") if host_clean else []

    if is_ip or len(parts) < 2:
        tld_len = 0
        dom_len = len(host_clean)
        subdom_cnt = 0
        dom_digits = 0
        dom_letters = 0
        dom_alnum = dom_len
    else:
        tld_len = len(parts[-1])
        dom_parts = parts[-2:]
        dom_str = ".".join(dom_parts)
        dom_len = len(dom_str)
        subdom_cnt = max(0, len(parts) - 2)

        dom_digits = sum(1 for c in dom_str if c.isdigit())
        dom_letters = sum(1 for c in dom_str if c.isalpha())
        dom_alnum = dom_digits + dom_letters

    dom_digit_ratio = dom_digits / dom_len if dom_len > 0 else 0.0
    dom_letter_ratio = dom_letters / dom_alnum if dom_alnum > 0 else 0.0

    is_https = 1 if url.lower().startswith("https://") else 0

    return [
        float(dom_len),
        float(is_ip),
        float(tld_len),
        float(subdom_cnt),
        float(has_www),
        entropy,
        dom_digit_ratio,
        dom_letter_ratio,
        float(is_https),
    ]

In [175]:
domain_features_names = [
    "dom_len",
    "is_ip",
    "tld_len",
    "subdom_cnt",
    "has_www",
    "entropy",
    "dom_digit_ratio",
    "dom_letter_ratio",
    "is_https",
]

### Dataset Generator

In [176]:
FEATURE_NAMES = domain_features_names
FEATURE_EXTRACTOR_FUNC = extract_domain_features

In [ ]:
test_urls = [
    'https://example.com',
] # paste here your URLs

In [178]:
test_urls_df = pd.DataFrame(test_urls, columns=["url"])

In [179]:
data = [FEATURE_EXTRACTOR_FUNC(url) for url in test_urls]

In [180]:
test_features_df = pd.DataFrame(data, columns=FEATURE_NAMES, index=test_urls_df.index)
test_df = pd.concat([test_urls_df, test_features_df], axis=1)

In [181]:
test_df.to_csv("../data/TestDataset.csv", index=False)

In [182]:
print("📥 Loading original dataset...")
df = pd.read_csv("../data/Dataset.csv")
output_urls_df = df[['url', 'label']].copy()

print(f"⚙️ Extracting features for {len(output_urls_df):,} URLs...")
features_data = []
errors = 0
batch = 20_000

for i, url in enumerate(output_urls_df['url']):
    try:
        features_data.append(FEATURE_EXTRACTOR_FUNC(url))
    except Exception as e:
        features_data.append([0.0] * 25)
        errors += 1
    
    if (i + 1) % batch == 0:
        print(f"   📦 Processed {i+1:,} / {len(output_urls_df):,}")

if errors > 0:
    print(f"⚠️ Failed to parse {errors} URLs. Filled with zeros.")

📥 Loading original dataset...
⚙️ Extracting features for 116,600 URLs...
   📦 Processed 20,000 / 116,600
   📦 Processed 40,000 / 116,600
   📦 Processed 60,000 / 116,600
   📦 Processed 80,000 / 116,600
   📦 Processed 100,000 / 116,600


In [183]:
output_features_df = pd.DataFrame(features_data, columns=FEATURE_NAMES, index=output_urls_df.index)
output_df = pd.concat([output_urls_df, output_features_df], axis=1)

In [184]:
output_path = "../data/Dataset_v2.csv"
output_df.to_csv(output_path, index=False)
print(f"\n✅ Saved new dataset to {output_path}")
print(f"📊 Shape: {output_df.shape}")
print(f"📋 Columns: {list(output_df.columns)}")

print("\n🔍 First 3 rows:")
print(output_df.head(3).to_string())
print("\n🔍 Feature statistics:")
print(output_df[FEATURE_NAMES].describe().to_string())


✅ Saved new dataset to ../data/Dataset_v2.csv
📊 Shape: (116600, 11)
📋 Columns: ['url', 'label', 'dom_len', 'is_ip', 'tld_len', 'subdom_cnt', 'has_www', 'entropy', 'dom_digit_ratio', 'dom_letter_ratio', 'is_https']

🔍 First 3 rows:
                          url  label  dom_len  is_ip  tld_len  subdom_cnt  has_www   entropy  dom_digit_ratio  dom_letter_ratio  is_https
0    https://www.rmit.edu.au/      0      6.0    0.0      2.0         1.0      1.0  3.095795              0.0               1.0       1.0
1  http://www.latrobe.edu.au/      0      6.0    0.0      2.0         1.0      1.0  3.235926              0.0               1.0       0.0
2     https://www.cqu.edu.au/      0      6.0    0.0      2.0         1.0      1.0  2.646439              0.0               1.0       1.0

🔍 Feature statistics:
             dom_len          is_ip       tld_len     subdom_cnt        has_www        entropy  dom_digit_ratio  dom_letter_ratio       is_https
count  116600.000000  116600.000000  116600.0000